In [1]:
import numpy as np
import pandas as pd

ID_SUM = 27     
SEED = 6       

# 1. Labeled DataFrame, keeping SECOM record IDs (row number + timestamp)
X = pd.read_csv("secom.data", sep=" ", header=None)
X.columns = [f"f{i}" for i in X.columns]
lab = pd.read_csv("secom_labels.data", sep=" ", header=None,
                  names=["label", "timestamp"], quotechar='"')
df = X.copy()
df["outcome"] = lab["label"].map({-1: "Pass", 1: "Fail"})
df["timestamp"] = lab["timestamp"]
df.index = pd.RangeIndex(1, len(df) + 1, name="record_id")

# 3. Select 5 usable features (not constant, not all missing), reproducibly
feat_pool = [c for c in X.columns
             if df[c].nunique(dropna=True) > 1]
rng = np.random.default_rng(SEED)
feats = sorted(rng.choice(feat_pool, size=5, replace=False).tolist())

# Median imputation using medians from the FULL dataset
medians = df[feats].median()
df[feats] = df[feats].fillna(medians)

# 2. Balanced sample: min(20 + ID_SUM, Pass count, Fail count) per outcome
counts = df["outcome"].value_counts()
n = int(min(20 + ID_SUM, counts["Pass"], counts["Fail"]))
sample = df.groupby("outcome").sample(n=n, random_state=SEED)
sample[["timestamp", "outcome"] + feats].to_csv("balanced_sample.csv")

# 4. Grouped statistics
stats = sample.groupby("outcome")[feats].agg(["count", "mean", "median", "std"])

# 5. Absolute mean difference and standardized mean difference (pooled SD)
g = sample.groupby("outcome")[feats]
mean, sd, cnt = g.mean(), g.std(ddof=1), g.count()
abs_diff = (mean.loc["Pass"] - mean.loc["Fail"]).abs()
pooled = np.sqrt(((cnt.loc["Pass"] - 1) * sd.loc["Pass"] ** 2 +
                  (cnt.loc["Fail"] - 1) * sd.loc["Fail"] ** 2) /
                 (cnt.loc["Pass"] + cnt.loc["Fail"] - 2))
smd = (abs_diff / pooled.where(pooled > 0)).fillna(0.0)   # zero-denominator safe

# 6. Rank and export
comp = pd.DataFrame({
    "pass_mean": mean.loc["Pass"], "fail_mean": mean.loc["Fail"],
    "pass_median": g.median().loc["Pass"], "fail_median": g.median().loc["Fail"],
    "pass_std": sd.loc["Pass"], "fail_std": sd.loc["Fail"],
    "abs_mean_diff": abs_diff, "pooled_sd": pooled, "abs_smd": smd,
}).sort_values("abs_smd", ascending=False)
comp.index.name = "feature"
comp["rank"] = range(1, len(comp) + 1)
comp.to_csv("feature_comparison.csv")

# NumPy-Pandas check
ok = True
for f in feats:
    p = sample.loc[sample.outcome == "Pass", f].to_numpy()
    q = sample.loc[sample.outcome == "Fail", f].to_numpy()
    ps = np.sqrt(((len(p)-1)*p.std(ddof=1)**2 + (len(q)-1)*q.std(ddof=1)**2)
                 / (len(p)+len(q)-2))
    s = 0.0 if ps == 0 else abs(p.mean() - q.mean()) / ps
    ok &= np.isclose(s, comp.loc[f, "abs_smd"])
    ok &= np.isclose(p.mean(), comp.loc[f, "pass_mean"])
    ok &= np.isclose(q.std(ddof=1), comp.loc[f, "fail_std"])

# Assessment data
print("Balanced Sample Size:", len(sample), f"({n} per outcome)")
print("Selected Features:", feats)
print("Highest-Ranked Feature:", comp.index[0])
print("Absolute SMD:", round(comp["abs_smd"].iloc[0], 4))
print("NumPy-Pandas Match:", "Yes" if ok else "No")

Balanced Sample Size: 94 (47 per outcome)
Selected Features: ['f171', 'f228', 'f289', 'f298', 'f564']
Highest-Ranked Feature: f228
Absolute SMD: 0.2653
NumPy-Pandas Match: Yes
